In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport

In [2]:
diab_df = pd.read_csv("../Dataset/diabetic_data.csv")

In [3]:
diab_profile = ProfileReport(diab_df, title = "Hospital Readmission Data Profile", minimal=True)
diab_profile.to_file("data_profile_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:01<00:00, 37.95it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [4]:
cols_to_drop = ['weight', 'payer_code', 'medical_specialty', 'examide', 'citoglipton']
diab_clean = diab_df.drop(columns=cols_to_drop)

In [5]:
diab_clean.shape

(101766, 45)

In [6]:
diab_clean = diab_clean[diab_clean['readmitted'] != '>30'].copy()
target_mapping = {'<30': 1, 'NO': 0}
diab_clean['target'] = diab_clean['readmitted'].map(target_mapping)
diab_clean = diab_clean.drop(columns=['readmitted'])

In [7]:
age_mapping = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35, '[40-50)': 45,
    '[50-60)': 55, '[60-70)': 65, '[70-80)': 75, '[80-90)': 85, '[90-100)': 95
}
diab_clean['age'] = diab_clean['age'].map(age_mapping)

In [8]:
diab_clean['A1Cresult'] = diab_clean['A1Cresult'].fillna('Not_Measured')
diab_clean['max_glu_serum'] = diab_clean['max_glu_serum'].fillna('Not_Measured')

In [9]:
admission_type_map = {
        1: "Emergency",       2: "Urgent",          3: "Elective",
        4: "Newborn",         5: "Not_Available",    6: "Not_Available",
        7: "Trauma_Center",   8: "Not_Mapped",
    }

discharge_disposition_map = {
    1:  "Discharged_Home",         2:  "Transfer_Short_Term",
    3:  "Transfer_SNF",            4:  "Transfer_ICF",
    5:  "Transfer_Other_Inpatient",6:  "Home_Health_Service",
    7:  "Left_AMA",                8:  "Home_IV_Provider",
    9:  "Admitted_Inpatient",      10: "Neonate_Transfer",
    11: "Expired",                 12: "Outpatient",
    13: "Hospice_Home",            14: "Hospice_Medical",
    15: "Swing_Bed",               16: "Outpatient_Referral",
    17: "Outpatient_This_Inst",    18: "Not_Available",
    19: "Expired_Home_Hospice",    20: "Expired_Medical_Hospice",
    21: "Expired_Unknown",         22: "Transfer_Rehab",
    23: "Transfer_Long_Term",      24: "Nursing_Medicaid",
    25: "Not_Mapped",              26: "Unknown",
    27: "Transfer_Federal",        28: "Transfer_Psychiatric",
    29: "Transfer_CAH",            30: "Transfer_Other",
}

admission_source_map = {
    1:  "Physician_Referral",      2:  "Clinic_Referral",
    3:  "HMO_Referral",            4:  "Transfer_Hospital",
    5:  "Transfer_SNF",            6:  "Transfer_Other",
    7:  "Emergency_Room",          8:  "Court_Law",
    9:  "Not_Available",           10: "Transfer_Critical",
    11: "Normal_Delivery",         12: "Premature_Delivery",
    13: "Sick_Baby",               14: "Extramural_Birth",
    15: "Not_Available",           17: "Not_Available",
    18: "Transfer_Home_Health",    19: "Readmission_Home_Health",
    20: "Not_Mapped",              21: "Unknown",
    22: "Transfer_Inpatient",      23: "Born_Inside",
    24: "Born_Outside",            25: "Transfer_Ambulatory",
    26: "Transfer_Hospice",
}

diab_clean["admission_type_id"]        = diab_clean["admission_type_id"].map(admission_type_map).fillna("Not_Available")
diab_clean["discharge_disposition_id"] = diab_clean["discharge_disposition_id"].map(discharge_disposition_map).fillna("Not_Mapped")
diab_clean["admission_source_id"]      = diab_clean["admission_source_id"].map(admission_source_map).fillna("Not_Available")

# Verify
for col in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
    print(f"\n{col} ({diab_clean[col].dtype}):")
    print(diab_clean[col].value_counts())


admission_type_id (object):
admission_type_id
Emergency        34681
Elective         13123
Urgent           12028
Not_Available     6124
Not_Mapped         236
Trauma_Center       21
Newborn              8
Name: count, dtype: int64

discharge_disposition_id (object):
discharge_disposition_id
Discharged_Home             38717
Transfer_SNF                 9038
Home_Health_Service          7540
Not_Available                2664
Expired                      1642
Transfer_Rehab               1474
Transfer_Short_Term          1460
Transfer_Other_Inpatient      834
Not_Mapped                    613
Transfer_ICF                  537
Left_AMA                      402
Hospice_Medical               365
Hospice_Home                  363
Transfer_Long_Term            268
Transfer_Psychiatric          105
Home_IV_Provider               70
Swing_Bed                      45
Nursing_Medicaid               32
Admitted_Inpatient             19
Outpatient_This_Inst            9
Expired_Home_Hospice     

In [10]:
print(f"Cleaned shape: {diab_clean.shape}")
print(f"\nTarget Distribution:\n{diab_clean['target'].value_counts(normalize=True) * 100}")

Cleaned shape: (66221, 45)

Target Distribution:
target
0    82.849851
1    17.150149
Name: proportion, dtype: float64


In [11]:
diab_clean.sample(10)

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,...,tolazamide,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,target
95026,356043302,130852994,AfricanAmerican,Female,75,Elective,Discharged_Home,Physician_Referral,1,1,...,No,No,No,No,No,No,No,No,No,0
60550,169467720,23886297,AfricanAmerican,Male,65,Elective,Discharged_Home,Physician_Referral,3,28,...,No,Steady,No,No,No,No,No,No,Yes,0
83282,260513214,101159181,Caucasian,Male,65,Emergency,Discharged_Home,Emergency_Room,4,59,...,No,No,No,No,No,No,No,Ch,Yes,0
97268,386627750,86047416,Caucasian,Male,55,Emergency,Discharged_Home,Physician_Referral,6,50,...,No,No,No,No,No,No,No,No,Yes,0
24928,83896254,21370131,Caucasian,Female,85,Elective,Home_Health_Service,Physician_Referral,3,11,...,No,No,No,No,No,No,No,No,No,0
81053,250590486,41647284,Caucasian,Male,75,Elective,Transfer_Short_Term,Physician_Referral,1,51,...,No,Up,No,No,No,No,No,Ch,Yes,0
86331,274529712,41543460,Caucasian,Male,75,Emergency,Discharged_Home,Emergency_Room,1,3,...,No,No,No,No,No,No,No,No,No,0
80024,246153972,73250406,Caucasian,Female,65,Emergency,Transfer_SNF,Emergency_Room,4,56,...,No,Steady,No,No,No,No,No,No,Yes,0
36925,113847198,64447182,Caucasian,Male,25,Emergency,Discharged_Home,Emergency_Room,2,59,...,No,Steady,No,No,No,No,No,Ch,Yes,0
72542,212421978,81875655,Caucasian,Female,75,Emergency,Transfer_SNF,Emergency_Room,6,61,...,No,No,No,No,No,No,No,No,No,1


In [12]:
def map_icd9_categories(val):
    # If the value is 'Unknown' (our replaced '?' marks)
    if val == 'Unknown':
        return 'Unknown'

    # Some codes start with 'V' or 'E' (supplemental classifications)
    if str(val).startswith('V') or str(val).startswith('E'):
        return 'Other'

    # Try to convert to float to map to numerical ICD-9 ranges
    try:
        val = float(val)
        if 390 <= val <= 459 or val == 785:
            return 'Circulatory'
        elif 460 <= val <= 519 or val == 786:
            return 'Respiratory'
        elif 520 <= val <= 579 or val == 787:
            return 'Digestive'
        elif np.floor(val) == 250:
            return 'Diabetes'
        elif 800 <= val <= 999:
            return 'Injury'
        elif 710 <= val <= 739:
            return 'Musculoskeletal'
        elif 580 <= val <= 629 or val == 788:
            return 'Genitourinary'
        elif 140 <= val <= 239:
            return 'Neoplasms'
        else:
            return 'Other'
    except:
        return 'Other'


for col in ['diag_1', 'diag_2', 'diag_3']:
    diab_clean[col] = diab_clean[col].apply(map_icd9_categories)

print("Diagnosis grouping complete!")
print("\nTop Primary Diagnoses (diag_1):")
print(diab_clean['diag_1'].value_counts(normalize=True) * 100)
print("\n")
print(diab_clean['diag_2'].value_counts(normalize=True) * 100)
print("\n")
print(diab_clean['diag_3'].value_counts(normalize=True) * 100)

Diagnosis grouping complete!

Top Primary Diagnoses (diag_1):
diag_1
Circulatory        29.594842
Other              18.007883
Respiratory        13.426255
Digestive           9.228190
Diabetes            8.213407
Injury              7.153320
Musculoskeletal     5.264191
Genitourinary       5.135833
Neoplasms           3.976080
Name: proportion, dtype: float64


diag_2
Circulatory        30.573383
Other              26.707540
Diabetes           12.982287
Respiratory        10.316969
Genitourinary       8.077498
Digestive           4.190514
Neoplasms           2.739312
Injury              2.647197
Musculoskeletal     1.765301
Name: proportion, dtype: float64


diag_3
Other              30.609625
Circulatory        29.162954
Diabetes           17.116927
Respiratory         6.987209
Genitourinary       6.309177
Digestive           3.793359
Injury              2.073360
Neoplasms           2.032588
Musculoskeletal     1.914800
Name: proportion, dtype: float64


In [13]:
save_path = r"C:\MCA\Project\Hospital_Readmission\Dataset\clean_data.csv"
diab_clean.to_csv(save_path, index=False)

print(f"Success! Clean data saved to {save_path}")

Success! Clean data saved to C:\MCA\Project\Hospital_Readmission\Dataset\clean_data.csv
